<h1>Chapter 2 - Large Language Models</h1>
<i>Exploring Large Language Model architecture</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 2 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1- Adding the `"Brain"`

Throughout this chapter, we covered the foundational structure of common LLMs, namely the Transformer. In this accompanying notebook, we will be adding *any* LLM to our `TinyAgent` as our first step towards autonomy.

![../images/ch2.png](../images/ch2.png)

To do so, we will have to consider which LLM we want to be using. This decision can be quite complex as it relates to your use case, your hardware (requirements), etc. Instead of providing you a full list of (quickly outdated) LLMs, we instead decided to allow readers to run almost *any* LLM by using the common `OpenAI` endpoint. Most LLMs these days are hosted either locally or online and accessing these servers can be done in many different ways. Most of them, however, expose an `OpenAI` API that allows us to query an LLM always in the same way. Likewise, this allows you to use any LLM whether it hosted by [`Ollama`](https://ollama.com/), [`vLLM`](https://github.com/vllm-project/vllm) or hosted in the cloud (like Gemini, Claude, and ChatGPT).

Let us explore various LLM inference engines for running your LLM that all create an `OpenAI` endpoint we can query:

## 2- **Inference Engines**

We split the inference engines by how they are typically used, namely either for local inference on your device on or using the cloud (which often is a propriety model).

### **Local**

As big fans of open-weight LLMs, we want users to be able to create and run Agents on their local devices. This setup can be a bit tricky since it requires setting up your environment, CUDA, Transformers, etc. Fortunately, there are a couple of packages that make this much easier.

#### 🔹 **Ollama**

[`Ollama`](https://ollama.com/) is arguably one of the most straightforward methods to run LLMs locally without needing to know much about how to setup your environment or offload layers to your GPU, Ollama does all of that for you and more. You can [download](https://ollama.com/) Ollama from their website. After installation, you can [download any model](https://ollama.com/search) using the following command

```bash
# Cannot reason and call tools natively
ollama pull gemma4:12b

# Has native reasoning and tool calling capabilities
ollama pull gemma4:e4b
```

After downloading the model, you can use the interface of the software to talk to your LLM:

![../images/ollama.png](../images/ollama.png)

As most local inference engines, it exposes a server with a REST API that we can query to run inference. `LiteLLM` takes care of that for us, so all you have to do is download `ollama` and a model that you want to run (we advise `gemma4:e4b` but more on that later!).

#### 🔹 **LM Studio**

Another popular method is using [`LM Studio`](https://lmstudio.ai/). Like Ollama, `LM Studio` can be downloaded and used to run models locally without having in-depth knowledge about setting up hardware-specific inference. After installing the software you first need to click on the "search" icon so we can begin downloading the model:

![../images/lmstudio1.png](../images/lmstudio1.png)

Then, search for your model (we are using gemma 4 with 4 billion parameters) and download like so:

![../images/lmstudio2.png](../images/lmstudio2.png)

Like Ollama, we can use the server endpoint to run our model. To check which adress it is using:

![../images/lmstudio3.png](../images/lmstudio3.png)

**NOTE**: Remember to also download the Gemma 4 E4B model as we will use that for native reasoning and tool calling capabilities.

#### 🔹 **Llama.cpp**

If you have experience with the terminal, Linux, setting up servers, etc. and want to have more control, then we advise the popular [`llama.cpp`](https://github.com/ggml-org/llama.cpp) package. This inference engines runs fully locally but requires you to install it yourself. Fortunately, [installation](https://github.com/ggml-org/llama.cpp/blob/master/docs/install.md) only requires a single line of code:

```sh
# Winget (Windows)
winget install llama.cpp

# Homebrew (Mac and Linux)
brew install llama.cpp
```

This will install `llama.cpp` on your device along with the `llama-server`. The latter is what `LiteLLM` will use to query your model. Before doing so, we first need to download the quantized model. Quantization is essentially a compression of a model's parameters in lower precision, which allows for much smaller models without too much of a drop in performance.

As always, we recommend downloading a the 4-bit quantized [Gemma 3](https://huggingface.co/unsloth/gemma-3-12b-it-GGUF/blob/main/gemma-3-12b-it-Q4_K_M.gguf) and [Gemma 4](https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/blob/main/gemma-4-E4B-it-Q4_K_M.gguf) models.

After doing so, you can start your server up with (for Gemma 3):

```bash
llama-server -m .\gemma-3-12b-it-Q4_K_M.gguf --port 8080
```

and the following for Gemma 4:

```bash
llama-server -m .\gemma-4-E4B-it-Q4_K_M.gguf --port 8080
```

The server and UI can then be accessed through your browser: http://localhost:8080

![../images/llamacpp.png](../images/llamacpp.png)

### **Cloud**

Although it is nice to run things locally, not everyone has access to powerful hardware capable enough to run these models. Instead, we can use cloud providers and use their (**free!**) tiers to run the Agent(s) throughout this book. Although we want to showcase the evolution of a non-reasoning/tool calling model to a reasoning/tool calling model, these large models are exceptionally strong and we can simply disable their reasoning and tool calling capabilities! So feel free to use a larger model as it should work without any isues throughout all examples.

#### 🔹 **Google**

Google has some amazing models to use (like the Gemini family of models) but also allow you to run their open-weight models (such as Gemma 4), which makes this a very interesting platform to use. We will be using the **Free Tier** which allows us to use some of these models with rate limits. To make sure we do not hit the rate limits, we advise to use their "flash" variants or their open-weight models. 

First, go to the [Gemini documentation](https://ai.google.dev/gemini-api/docs/api-key) and click on "Create or view a Gemini API Key":

![../images/google1.png](../images/google1.png)

Then, click on "Create API Key":

![../images/google2.png](../images/google2.png)

Before you can actually create a key, you may need to first create a project if you hadn't already done so. Then select it and create the key:

![../images/google3.png](../images/google3.png)

Finally, copy your API key to use throughout this book.

#### 🔹 **OpenAI**

You can also use one of OpenAI's models, which do not have a Free tier unfortunately. That said, if you already happen to have an API Key, you can find it [here](https://platform.openai.com/api-keys) and simply use that for one of their models.

#### 🔹 **Claude**

Although we haven't tried it ourselves yet, free credits are given to new users to test the API. We are not sure whether this is sufficient for testing the entire book, so be aware. You can find more about their pricing [here](https://platform.claude.com/docs/en/about-claude/pricing).

## 3 - **Which model should I use?**

The models that we are going to use throughout this book are Gemma 3 and Gemma 4 variants. As mentioned before, we like to showcase the evolution of these models as that will help you understand what these newer models (like Gemma 4) can "just do". They are trained to perform reasoning and tool calling which takes away some of the intuition behind these techniques.

![../images/evolution.png](../images/evolution.png)

The Gemma 3 and 4 models come in various sizes.

Gemma 3 has four sizes (1B, 4B, 12B, and 27B) and we have been succesfully testing the model with 12 billion parameters (12B) as a quantization (compression) of 4 bits. This model can be run locally if you have 12GB of (V)RAM available. All of these models are great contenders, but if you want more performance than the E4B, we would advise using [27B](https://ollama.com/library/gemma3:27b) instead.

Gemma 4 has four sizes (E2B, E4B, 26B A4B, and 31B) and we have been succesfully testing the model with 4 billion parameters (E4B) at a quantization (compression) of 4 bits. This model can be run locally if you have 12GB of (V)RAM available. All of these models are great contenders, but if you want more performance than the E4B, we would advise using [26B A4B](https://ollama.com/library/gemma4:26b) instead.

That said, all examples can be done with a sufficiently capable model! There is no way we are going to limit you to only use specific models. We have implemented all examples so that you can use any model with any server with an `openai` endpoint :)

Although we have not tested all of them, these are models that are at the time of writing (April 2026) considered one of the best in their size category:

* Gemma 4
* Qwen 3.5
* GPT-OSS
* GLM-4.7 Flash
* Phi-4

Larger, open-weight models include:

* Kimi-K2
* DeepSeek V3.2

## 4 - **The LLM Wrapper**

Now that we have explored all various types of models and inference backends, let's create our `LLM` class that we use throughout this book:

Before doing that, let's explore this `"Brain"` (the `LLM`) that we want to add to our `TinyAgent`:

In [1]:
from openai import OpenAI

class LLM:
    def __init__(self, model: str, client: OpenAI, **kwargs):
        """Initialize the LLM with the given model."""
        self.model = model
        self.client = client
        self.kwargs = kwargs

    def generate(self, messages: list[dict]):
        """Generate a response from the LLM given a list of messages."""
        response = self.client.chat.completions.create(
            model=self.model, 
            messages=messages, 
            **self.kwargs
        )
        return response

We are using the [openai](https://github.com/openai/openai-python) package which uses a very specific formatting structure for how the response of the LLM is parsed. It includes standardized fields, parameters, and response fields so that you know you can always get a reply back in the same one. As OpenAI were a major start in this AI-boom, their API has become one of the standards of accessing LLMs that are hosted on servers. 

Fortunately, this makes the implementation of your `LLM` rather straightfoward. You can create the `LLM` class and use `self.client.chat.completions.create` to run your model. In OpenAI's `response` you can find not only the output of the model but also its intermediate reasoning (if any) and other metadata. We are going to show `Gemma 4` first since that has additional fields (like `reasoning` and `tool_calls`) to explore.

In [2]:
from openai import OpenAI

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma4:e4b", client=client)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-4-E4B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-4-e4b-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

We can then use the `messages` structure to talk to the LLM. These `messages` is a common format for structuring multiple turns of question/responses between you (`user`) and the LLM (`assistant`). To ask a question, we use the `user` role:

In [3]:
# Formatting the query
messages = [
    {
        "role": "user", 
        "content": "Hi! How's life?"
    }
]

# Generating a response
response = llm.generate(messages=messages)
response

ChatCompletion(id='chatcmpl-168', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Hi there! Life is going really well, thanks for asking! 😊 I'm currently processing a lot of information and helping people with all kinds of questions, so it keeps me busy and interesting!\n\nHow about you? How's life treating you today? Anything fun or interesting going on?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1778420462, model='gemma4:e4b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=61, prompt_tokens=23, total_tokens=84, completion_tokens_details=None, prompt_tokens_details=None))

The content of the model's output can be accessed by the `ChatCompletion.choices` field. Although a model could technically give back multiple answers in parallel, it is often just a single response. As such, most people will use `ChatCompletion.choices[0]`.

In there, you will find, among others:

* `content` -- The main answer of the LLM
* `reasoning` or `reasoning_content` -- The thinking behavior of the LLM. We explore this in more detail in Chapter 3.
* `tool_calls` -- These are the cools a model might use. 

Let's print `responses.choices[0]` nicely with [`rich`](https://github.com/Textualize/rich) which is a great package for styling interfaces:

In [4]:
from rich import print
print(response.choices[0])

Choice(
    finish_reason='stop',
    index=0,
    logprobs=None,
    message=ChatCompletionMessage(
        content="Hi there! Life is going really well, thanks for asking! 😊 I'm currently processing a lot of 
information and helping people with all kinds of questions, so it keeps me busy and interesting!\n\nHow about you? 
How's life treating you today? Anything fun or interesting going on?",
        refusal=None,
        role='assistant',
        annotations=None,
        audio=None,
        function_call=None,
        tool_calls=None
    )
)

When we look what is in this `response.choices[0]` we can see that it returns a `Choice` with a `message` where you can find the answer (`content`) but also whether there was any reasoning (`reasoning`) or tool calls (`tool_calls`). For the purpose of this book, we are not going to be using the tool and reasoning fields immediately! The reason for that is two-fold and is important for the philosophy behind understanding what is happening under the hood. 

1) Using `reasoning` and `tool_calls` is a bit "magical" and does not teach you how these are actually created and used. Instead, we are going to show you how to do this explicitly.

2) Some models actually do not support `reasoning` and `tool_calls` but with proper prompting can still be used as Agents! So instead of blindly relying on these fields, we want to show you how nudge the model to do explicit reasoning and tool calling with prompting.

That said, of course we are going to show you how to use those fields! It's important to see what is happening under the hood (through prompting) and how that behavior eventually evolved to models being learned how to do this (native `reasoning` and `tool_calls`).

There is a tiny problem though with having three different backends, they all give back responses in slightly different ways! To use all of them, we are going to wrap their answers into a `Response` dataclass where we will track their `content`, `reasoning`, `tool_call`, and other `metadata` the model may provide:

In [5]:
from dataclasses import dataclass

@dataclass
class Response:
    """Structured response from LLM calls."""

    content: str = ""
    reasoning: str | None = None
    tool_call: dict | None = None
    metadata: dict | None = None

For now, we are only interested in using the `content` of the model as we want to show you how to have a non-reasoning model still show reasoning traces! To do that, we are adding a parameter to the `LLM` that can disable thinking.

In [18]:
from openai import OpenAI

class LLM:
    def __init__(self, model: str, client: OpenAI, think: bool = False, **kwargs):
        """Initialize the LLM with the given model."""
        self.model = model
        self.client = client
        self.think = think
        self.kwargs = kwargs

    def generate(
        self, messages: list[dict], tools: list | None = None
    ) -> Response:
        """Generate a response from the LLM given a list of messages."""
        # Enable/Disable thinking
        if self.think:
            extra_body = None
        else:
            extra_body = {
                "chat_template_kwargs": {"enable_thinking": False},
                "reasoning_effort": "none",
            }

        # Generate a response
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=tools if tools else None,
            extra_body=extra_body,
            **self.kwargs,
        )

        # Extract message, tool_call, and metadata
        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", None) or getattr(
            message, "reasoning", None
        )
        has_tool_call = hasattr(message, "tool_calls") and message.tool_calls
        tool_call = message.tool_calls[0].model_dump() if has_tool_call else None
        metadata = {
            "model": response.model,
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        }

        # Format as Response dataclass
        return Response(
            content=message.content,
            reasoning=reasoning,
            tool_call=tool_call,
            metadata=metadata,
        )

Note we are going to skip adding the tool call right now but we will circle back in Chapter 5!

Let's explore these steps in more detail and let's zoom in a bit on what we did with the `openai` function:

In [19]:
from illustrated_agents.chapters.ch2 import llm_annotated; llm_annotated

Finally, we can call the LLM with our updated class and extract the response only:

In [20]:
# Re-initialize the LLM with the updated class
llm = LLM(model="gemma4:e4b", client=client)

# Generate a `Response` dataclass
response = llm.generate([{"role": "user", "content": "Hi! How's life?"}])
print(response)

Response(
    content="I'm doing great! I'm busy processing information and helping users like you, which is always 
interesting. 😊\n\nHow is life treating *you* today? Anything exciting or anything I can help you with?",
    reasoning=None,
    tool_call=None,
    metadata={'model': 'gemma4:e4b', 'prompt_tokens': 16, 'completion_tokens': 45}
)

We have disabled reasoning for Gemma 4 E4B. However, these models shine when reasoning is enabled. Instead of trying to limit this model, we are going to use Gemma 3 12B throughout all prompting-based examples instead. For the purpose of this book, we believe it is much more interesting to enhance a model rather than try to limit it. That said, if you want to use a more capable model throughout this book, you can! Just make sure to disable thinking as we are going to try to enable that through prompting.

Note that in the end we are working towards native tool calling and reasoning, so no prompt-based techniques. As such, we hope to show you this evolution from prompt-based techniques to native agentic capabilities!

## 5 - The `Trajectory`

Before we explore how to update your `agent.py`, there is something else to explore first. A vital component of Agentic Harnesses (which is the framework that we are creating) is making sure you can easily debug what is happening. An Agent might run for several turns at a time and run into an incorrect tool usage. To easily debug that we want to keep track of everything the model has done so far. To do that, we track each `Step` the agent has taken (including `observation` from the output of a tool):

In [10]:
from dataclasses import dataclass

@dataclass
class Step:
    """A single step in an agent's trajectory."""

    thought: str = ""
    action: dict | None = None
    observation: str | None = None
    answer: str | None = None
    metadata: dict | None = None

Note that `Step` is very much like the `Response` object that we saw before but with a couple of additional fields, namely `answer` and `observation`. The `answer` is the final answer of the model and represents that the Agent has reached the end of its turn. The `observation` is the output of tool usage. For example, it can search the web for information about flamingos and get returned a bunch of information (`observation`). Finally, instead of having a `tool_call` field, we now use `action`. This is on purpose as it is part of the THOUGHT/ACTION/OBSERVATION loop that we will explore in Chapter 6.

An Agent might need various steps to get to a final answer and use many different actions to reach that call. All these steps together is what we call an Agent's `Trajectory`. We want to make sure we can easily track that information. The `Trajectory` of your Agent is as follows and allows you to add your query with `initialize` and add a full step with `add` where the potential reasoning, action, observation, answer, and metadata is tracked:

In [11]:
class Trajectory:
    """Records agent execution as a sequence of runs."""
    def __init__(self) -> None:
        self.runs: list[dict] = []

    def initialize(self, query: str) -> None:
        """Register a new run with the given query."""
        self.runs.append({"query": query, "steps": []})

    def add(self, response: Response, observation: str | None = None) -> None:
        """Record a step from a Response, optionally with an observation."""
        # Add THOUGHT
        step = Step(
            thought=response.reasoning or "",
            metadata=response.metadata,
        )

        # Add ACTION/OBSERVATION or ANSWER
        if observation is not None:
            step.action = response.tool_call
            step.observation = observation
        else:
            step.answer = response.content
            
        self.runs[-1]["steps"].append(step)

The trajectory is meant only to track a `Step` easily:

In [12]:
from illustrated_agents.chapters.ch2 import trajectory_add_annotated; trajectory_add_annotated

## 5 - Updating `agent.py`

We update our `TinyAgent` to have a brain and actually be able to answer:

In [13]:
class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM):
        self.llm = llm
        self.memory = None  # Chapter 4: Add Memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning

        self.trajectory = Trajectory()

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.trajectory.initialize(task)
        return self._step(task)

    def _step(self, task: str) -> str:
        """Perform a single step."""
        messages = [{"role": "user", "content": task}]
        response = self.llm.generate(messages)
        self.trajectory.add(response)
        return response.content

    def _execute_action(self, action: str) -> str | None:
        """Execute a tool action."""
        # Placeholder - will be implemented in later chapters
        return f"Executed action: {action}"

Here is a nicer overview of the changes that we made to `agent.py` (red is removed and green is added code):

In [14]:
from illustrated_agents.chapters.ch2 import tinyagents_diff; tinyagents_diff

We can then run our agent as follows:

In [15]:
agent = TinyAgent(llm=llm)
response = agent.run("What is 2 + 2?")
print(response)

2 + 2 is **4**.

Great, it gives back an answer! Throughout these chapters we also will be looking at the `Trajectory` quite often to debug and see if the `TinyAgent` did as expected. We can access the full trajectory with:

In [16]:
print(agent.trajectory.runs)

[
    {
        'query': 'What is 2 + 2?',
        'steps': [
            Step(
                thought=(None,),
                action=None,
                observation=None,
                answer='2 + 2 is **4**.',
                metadata={'model': 'gemma4:e4b', 'prompt_tokens': 17, 'completion_tokens': 9}
            )
        ]
    }
]

Note though that this can get large quite quickly if we have dozens of tool calls, long reasoning, and more! So, we decided to create a small helper class that visualizes these components. This `TrajectoryViewer` uses basic HTML to gives a bit more control over the what we see and don't see in the Trajectory. After you run the cell, you can click on each run and step block to see same information as above, just a bit more interactive:

In [17]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

Our "Agent" is still nothing more than the LLM and has no additional behavior/capabilities that we can showcase yet. For that, we need to add more modules, such as **reasoning** (Chapter 3), **memory** (Chapter 4), **tools** (Chapter 5), and **planning** (Chapter 6).

---

⚠️ **IMPORTANT**: LLMs are stocastich and will not always produce the same output. So it might happen, especially with smaller models, that you do not get the output you are looking for in the examples throughout the book. They are efficient and small examples, so you can simply rerun the examples once and twice and it should work. Note that larger models, like Gemini 3, is unlikely to have that problem since it is quite adept.

---

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered the basic steps in querying an LLM, what it outputs and how you could connect it to your `TinyAgent`.

In [ ]:
from illustrated_agents.chapters.ch2 import what_we_built; what_we_built

# What's Next

In the next chapter, we will explore how reasoning works from the perspective of both prompting and native reasoning.